> **💾 First: save your own copy** · This notebook opens from the course website, so Colab does not keep your changes. Before you do anything else, go to **File → Save a copy in Drive**. Colab opens your copy in a new tab: work in that tab and close this one. You find your copy later in Google Drive, in the folder *Colab Notebooks*.

> **⚠️ Important** · Please run the cell below to set up the environment.

In [ ]:
%%capture
#Keep cell

import os
import nltk

try:
  nltk.data.find('tokenizers/punkt_tab')
except LookupError:
  nltk.download('punkt')
  nltk.download('punkt_tab')

try:
  nltk.data.find('corpora/stopwords')
except LookupError:
  nltk.download('stopwords')


try:
  import gensim
except:
  !pip install gensim
  import gensim

if not(os.path.exists('sotu_all_new.csv')):
  print('Downloading sotu_all_new.csv\n')
  !wget https://raw.githubusercontent.com/goto4711/cdai/main/sotu_all_new.csv

if not(os.path.exists('hate-vocab-eng.csv')):
  print('Downloading hate-vocab-eng.csv')
  !wget https://raw.githubusercontent.com/goto4711/cdai/main/hate-vocab-eng.csv

![Cultural Data Analysis · Week 4: Cultural Data Forms: Text](https://goto4711.github.io/cda-teaching-tools/assets/notebooks/week4-banner.png)

# Cultural Data Forms - Texts

*Lecture notebook* · Code-along: [goto4711.github.io/cda-teaching-tools/code-along/week4](https://goto4711.github.io/cda-teaching-tools/code-along/week4/) · Tools: [term frequency](https://goto4711.github.io/cda-teaching-tools/tools/text/), [from words to embeddings](https://goto4711.github.io/cda-teaching-tools/tools/embeddings/)

---

> **Goals** · This notebook is about one question: **how does a computer read a text?** It cannot read the way we do. It turns a text, step by step, into something it can count and calculate with:
> 1. **Tokens:** cutting a text into words.
> 2. **Cleaning:** lowercasing, and removing punctuation, numbers and stopwords.
> 3. **Counting:** turning documents into rows of numbers, and weighting them with tf-idf.
> 4. **Reading with a trained model:** finding people, places and organisations, and the grammar of a sentence.
> 5. **Methods built on counting:** dictionary-based sentiment analysis, topic models, and word embeddings, the step towards today's language models.
>
> At every step, we also ask: **what has the computer lost?**

Texts are *unstructured data*: information that "does not have a pre-defined data model" (https://en.wikipedia.org/wiki/Unstructured_data). So much of what humans produce is text that text analysis is one of the oldest fields of computational analysis, with very established methods. Many of your teachers come from the Institute for Logic, Language and Computation (https://en.wikipedia.org/wiki/Institute_for_Logic,_Language_and_Computation).

## The State of the Union Addresses

We work with every US State of the Union Address (SOTU) from 1790 to 2022, taken from http://stateoftheunion.onetwothree.net/index.shtml. They let us follow how words gain and lose prominence over more than 200 years.

Run the cell below to load the speeches into the dataframe `sotu_df`.

In [ ]:
#Keep cell
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [16, 8]

sotu_df = pd.read_csv('sotu_all_new.csv')
sotu_df.tail()

You cannot see the whole text in the `speechtext` column. Print the first 100 characters of the first speech with `sotu_df.loc[0, 'speechtext'][:100]`.

We will use the years as names of speeches, not as numbers, so we turn them into strings: type `sotu_df['year'] = sotu_df['year'].astype(str)`.

> **💡 Remember** · The `date` column is still a number like 17900108. Fighting with different formats of date and time is a key part of analysing culture and society computationally: for some past events, we only know an approximate time span. Week 2 showed you how `pd.to_datetime()` turns such numbers into dates.

### Our running example

To see what the computer does to a text, we follow one sentence through every step. It comes from Barack Obama's first address to Congress, in 2009. Read it carefully: it is a political statement, and every word matters. Run the cell below.

In [ ]:
#Keep cell
example = "And that is why I can stand here tonight and say without exception or equivocation that the United States of America does not torture."
print(example)

---

# Tools, tools, tools

## Step 1: Tokens, with NLTK

The Natural Language Toolkit (NLTK; https://www.nltk.org) is a classic Python package for natural language processing. It is in parts outdated, but it is still where much text analysis in Python begins.

Our research question: how did the policies of George W. Bush (his last address, 2008) and Barack Obama (his first, 2009) differ, at the height of the financial crisis? First, we get the two speeches as two strings. Type in:
```
import nltk
sotu_2008_text = sotu_df.loc[sotu_df['year']=='2008','speechtext'].item()
```
and add a line for `sotu_2009_text`. `item()` gets the content of the one cell you select.

The first step in how a computer reads a text is to *tokenize* it: to break it into its units, usually words (https://www.datacamp.com/blog/what-is-tokenization). Let's see what that does to our example. Run the cell below.

In [ ]:
#Keep cell
from nltk.tokenize import word_tokenize

example_tokens = word_tokenize(example)
print(example_tokens)
print(len(example_tokens), 'tokens')

> **🔍 What has the computer lost?** · The sentence is now a list of separate pieces. "United States of America" has become four tokens, and the computer no longer knows that they name one country. The full stop is a token, just like a word.

Now tokenize both speeches into lists of words, `corpus_2008` and `corpus_2009`. Type in `corpus_2008 = word_tokenize(sotu_2008_text)` and add the line for 2009.

## Step 2: Cleaning

> **💡 Remember** · Pay attention to the steps of this so-called *pipeline*. You will see them again and again, in different orders.

First, we make all words **lowercase**, so that 'House' and 'house' count as the same word. Second, we remove **numbers** and **punctuation**, which NLTK keeps as tokens. Python's `isalpha()` checks whether a token consists only of letters, so it removes both.

Type in `corpus_2008 = [word.lower() for word in corpus_2008 if word.isalpha()]` and add the same line for 2009. Do you remember list comprehensions from the code-along? (If not: https://www.w3schools.com/python/python_lists_comprehension.asp.)

Third, we remove **stopwords** such as 'the', 'a' or 'or', which are thought to carry little meaning. NLTK has lists of them. Run the cell below to see the first English and Dutch ones.

In [ ]:
#Keep cell
from nltk.corpus import stopwords
stopword_list = list(stopwords.words('english'))
print(stopword_list[:10])
print(list(stopwords.words('dutch'))[:10])

Much of text analysis depends on the language, and there are most tools and resources for English. This is why we focus on it for now.

Remove the stopwords: type in `corpus_2008 = [word for word in corpus_2008 if word not in stopword_list]` and add the line for 2009.

Now the same cleaning for our example sentence. Run the cell below and compare the result with the original sentence.

In [ ]:
#Keep cell
# lowercase, and keep only words made of letters
example_clean = [word.lower() for word in example_tokens if word.isalpha()]
# remove the stopwords
example_clean = [word for word in example_clean if word not in stopword_list]
print(example_clean)

Something is missing. Is 'not' a stopword? Check it with `'not' in stopword_list`.

> **🔍 What has the computer lost?** · Read the cleaned sentence out loud: "united states america torture". "Not" is on NLTK's list of stopwords, so cleaning has turned the statement into its opposite. Stopword lists were made for search engines, where small words rarely matter. For meaning, they can matter a lot. Who decided what goes on the list?

### A first look: word clouds

A word cloud (https://en.wikipedia.org/wiki/Tag_cloud) shows the most frequent words of a text, bigger the more often they appear. It needs a single text, so we join each list of words back together: `text_2008 = ' '.join(corpus_2008)` means "join all elements of the list, with a space in between". Type it in and add the line for 2009.

> **💡 Remember** · These are the steps of almost every text analysis: split the text into words, clean the words, and join them back into a text.

Run the cell below to show both word clouds next to each other.

In [ ]:
#Keep cell
from wordcloud import WordCloud

plt.subplot(1,2,1)
plt.imshow(WordCloud(background_color = 'white').generate(text_2008))
plt.title('Bush', fontsize = 30)
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(WordCloud(background_color = 'white').generate(text_2009))
plt.title('Obama', fontsize = 30)
plt.axis("off")
plt.show()

Obama concentrated more on the economy, while Bush's favourite topic was the war on terror. The times changed a lot between them, which is why we chose these two.

> **🔍 What has the computer lost?** · A word cloud only shows how often a word appears. The size of a word says nothing about whether it was praised, criticised or denied.

---

## Step 3: Counting

### The document-term matrix

To compare all 236 speeches, the computer needs numbers. We count the words of each speech and build a *document-term matrix* (DTM): one row per document, one column per word, and in each cell how often the word appears in the document (https://en.wikipedia.org/wiki/Document-term_matrix). This is the main way computers read texts: by counting.

sklearn's `CountVectorizer` does the hard work, including the cleaning we did by hand: it lowercases, removes English stopwords, and keeps only words of at least three letters with the regular expression `'[a-zA-Z]{3,}'` (you met regular expressions in the code-along; I asked Gemini to help me write this one). As in Week 3, we then *fit* and *transform*. Type in:
```
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(lowercase = True, stop_words = 'english', token_pattern = '[a-zA-Z]{3,}')
vectors = vectorizer.fit_transform(sotu_df['speechtext'])
```

Run the cell below to turn the result into a dataframe `dtm_df`, with the years as row names and the words as column names.

In [ ]:
#Keep cell
dtm_df = pd.DataFrame(vectors.toarray(),
                      columns = vectorizer.get_feature_names_out(),
                      index = sotu_df['year'])

dtm_df.head()

What does our example sentence look like as a row of this matrix? Run the cell below. It shows only the columns where the sentence has a count.

In [ ]:
#Keep cell
example_row = pd.DataFrame(vectorizer.transform([example]).toarray(),
                           columns=vectorizer.get_feature_names_out())
print((example_row.iloc[0] > 0).sum(), 'of', example_row.shape[1], 'columns are not 0')
example_row.loc[:, example_row.iloc[0] > 0]

> **🔍 What has the computer lost?** · The sentence has become one row of numbers, almost all of them 0. The order of the words is gone: this *bag of words* would be the same in any other order. The vectorizer has its own stopword list, and "not" is missing again.

Now compare Bush and Obama. Select their two rows and sort the columns by Bush's counts. Type in:
```
dtm_df_2008_2009 = dtm_df.loc[['2008', '2009'], :]
dtm_df_2008_2009 = dtm_df_2008_2009.sort_values(by='2008', ascending=False, axis = 1)
dtm_df_2008_2009.iloc[:,:15]
```
`axis = 1` sorts the columns (the words) by their values in the row for 2008.

Iraq, for instance, almost disappears in Obama's speech.

### Word trends

With a count for every word in every year, we can follow a word through history. Run the cell below to make the figures smaller again; we enlarged them for the word clouds.

In [ ]:
#Keep cell
plt.rcParams['figure.figsize'] = [8, 4]

Plot how often 'america' and 'citizen' appear in each speech with `dtm_df[['america', 'citizen']].plot()`. Then plot the trend of 'history' in a second line.

> **💡 Remember** · Pandas' built-in plots are quick, but not pretty: good for exploring data, not for presenting it.

### Weighting: tf-idf

Counting favours words that are frequent everywhere, like 'people' or 'america'. Tf-idf weights each count: a word scores high if it is frequent in one speech (term frequency, TF) but appears in few other speeches (inverse document frequency, IDF). You know the idea from the code-along.

The cell below builds the same table with tf-idf weights instead of counts, and shows the ten words with the highest weight in Bush's 2008 speech. Run it.

In [ ]:
#Keep cell
from sklearn.feature_extraction.text import TfidfVectorizer

# The same settings as our CountVectorizer, but with tf-idf weights
tfidf = TfidfVectorizer(lowercase = True, stop_words = 'english', token_pattern = '[a-zA-Z]{3,}')
vec = tfidf.fit_transform(sotu_df['speechtext'])

tfidf_df = pd.DataFrame(vec.toarray(), columns = tfidf.get_feature_names_out(), index = sotu_df['year'])
tfidf_df.loc['2008'].sort_values(ascending=False).head(10)

Compare this list with the top counts for 2008 above. Then show the ten most distinctive words of Obama's 2009 speech. Why is 'don' on his list? Look at the `token_pattern`.

> **🔍 What has the computer lost?** · Tf-idf finds that 'iraq' is distinctive for 2008. But the computer does not know what Iraq is, or that the speech was given during a war there. It only knows that the word is frequent in this speech and rare in the others.

---

## Step 4: Reading with a trained model

So far, the computer has only cut and counted. *Information extraction* (https://en.wikipedia.org/wiki/Information_extraction) goes further: a model trained on many annotated texts finds the names of people, places, organisations and dates. Who, what, where and when are key questions in social and cultural research.

We use spaCy (https://spacy.io) with `en_core_web_sm`, a small English model trained on web texts: blogs, news and comments. Try the demo at https://explosion.ai/demos/displacy-ent first. Then load spaCy and the model by typing:
```
import spacy
nlp = spacy.load('en_core_web_sm')
```

`nlp` reads a text and extracts its linguistic features. Let's start with our example. Run the cell below.

In [ ]:
#Keep cell
spacy.displacy.render(nlp(example), style='ent', jupyter=True)

"United States of America" is one entity again: a GPE, a geopolitical entity such as a country or city. "Tonight" is a TIME. The model has learned from other texts that these words belong together.

spaCy can also read the grammar of a sentence with *dependency parsing* (https://spacy.io/usage/visualizers): which word is the subject, which the verb, and how the words relate. Run the cell below for a short version of our sentence.

In [ ]:
#Keep cell
spacy.displacy.render(nlp("The United States does not torture."), style='dep', jupyter=True)

> **🔍 What has the computer lost?** · Here, the computer gains something back: the parse attaches "not" to "torture" as a negation (neg). But the model only knows its training data, English web texts of today. How well will it read a speech from 1790?

Now read Bush's whole 2008 speech. Type in `doc = nlp(sotu_2008_text)`, then list its first 20 entities with their labels: `[(entity.text, entity.label_) for entity in doc.ents][:20]`.

Which places does Bush mention? Show only the 'GPE' entities with `[ent for ent in doc.ents if ent.label_ in ['GPE']][:20]`.

Entities help us understand content, and they link texts: two documents about the same place may be connected. Want to see Bush's places on a map? There is an optional map at the end of this notebook, for at home.

---

## Step 5: Three methods built on counting

Sentiment analysis, topic models and word embeddings all start from the steps we have just done. Each reads the texts in a different way.

### Sentiment analysis with a dictionary

Sentiment analysis asks whether a text is positive or negative; hate-speech detection asks whether it contains hate speech. It is a bit strange to ask this of the SOTU speeches, but it prepares the workshop, where we detect hate speech in Tweets.

We use the simplest approach, a *dictionary*: a list of hate-speech words, counted in each text. Gary King, for instance, has consistently warned against measuring sentiment like this (http://gking.harvard.edu/). We use it because it shows clearly how the computer reads. Our list is a reduced, older version of the Hatebase dictionary (https://web.archive.org/web/2023/https://hatebase.org/how_it_works; the project has since closed), for demonstration only.

Run the cell below to load the list `neg_words` and print five of its words. They can be hard to understand without context.

In [ ]:
# Keep cell

neg_words = list(set(list(pd.read_csv('hate-vocab-eng.csv')['word'])))

neg_words[:5]

Why might such a list depend on how language is used? Does it change over time? What does this mean for a historical analysis of SOTU?

Now a function, `hate_score_`, that tokenizes and lowercases a text and adds 1 to the score for every word that is in `neg_words`. Type it in:
```
def hate_score_(text):
    score = 0
    words = [word.lower() for word in word_tokenize(text)]
    for word in words:
        if word in neg_words:
            score += 1
    return score
```

Apply it to every speech with `sotu_df['hate_score'] = sotu_df['speechtext'].apply(hate_score_)`. This might take a little while.

Longer speeches will contain more of any words, so we *normalise*: we divide each score by the length of the speech. `str.split()` splits each text into words and `len` counts them. Complete the cell below.

In [ ]:
#Complete cell
sotu_df['hate_score'] = sotu_df['speechtext'].apply(___)
speech_lengths = sotu_df['speechtext'].str.split().apply(___)
sotu_df['hate_score_normalized'] = sotu_df['hate_score'] / ___
sotu_df.plot(x='year', y='hate_score_normalized', kind='scatter')

The scores are very low: there is not much hate speech in the State of the Union, fortunately. Which speeches score highest? Sort by the normalised score and keep the top three: `top_hate_three_df = sotu_df.sort_values(by=['hate_score_normalized'], ascending=False).head(3)`.

Which words made the top speech score so high? Run the cell below.

In [ ]:
#Keep cell
top_speech = top_hate_three_df.index[0]
top_words = [word.lower() for word in word_tokenize(sotu_df.loc[top_speech, 'speechtext'])]
pd.Series([word for word in top_words if word in neg_words]).value_counts().head(10)

> **🔍 What has the computer lost?** · The top speech is Abraham Lincoln's annual message of December 1862, in the middle of the Civil War, which discusses slavery and emancipation. The dictionary finds words such as 'colored' and 'property' there. The computer reads through someone else's list of words: it cannot tell whether a word insults, describes or means something else entirely, and it does not know that words change their meaning over time. Use dictionary methods with great care, and never on their own.

### Topic modelling: words that appear together

Topic modelling summarises a collection of texts into a number of topics, each a list of words that often appear together (http://journalofdigitalhumanities.org/2-1/topic-modeling-and-digital-humanities-by-david-m-blei/). Topic models have been the most popular method in the data projects later in the semester, so pay extra attention. But there is no guarantee that the topics are what people would consider topics, and interpreting them can become like 'reading tea leaves' (https://proceedings.neurips.cc/paper/2009/file/f92586a25bb3145facd64ab20fd554ff-Paper.pdf).

We use Latent Dirichlet Allocation (LDA; https://en.wikipedia.org/wiki/Latent_Dirichlet_allocation) on our word counts, `vectors`. As with k-means, we have to choose the number of topics in advance. Type in:
```
from sklearn.decomposition import LatentDirichletAllocation
lda = LatentDirichletAllocation(n_components=5, random_state=4711).fit(vectors)
```

Run the cell below to print the five words most associated with each topic.

In [ ]:
#Keep cell
def print_top_words(model, feature_names, n_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print("Topic #%d:" % topic_idx)
        print(" ".join([feature_names[i]
                        for i in topic.argsort()[:-n_top_words - 1:-1]]))

    print()


print("\nTopics in LDA model:")
feature_names = vectorizer.get_feature_names_out()
print_top_words(lda, feature_names, 5)

Give each of the five topics a title. That is not always easy: remember our struggles with interpreting clusters in Week 3? If you are interested, I have used topic models in https://www.nature.com/articles/s41599-024-02668-4.

> **🔍 What has the computer lost?** · The topics are lists of words that appear in the same speeches. The computer gives them no names and no meaning: you do. And you decided in advance how many there are.

### Word embeddings: meaning from neighbours

Our last tool is the key to today's language models. Word embeddings represent each word as a vector of numbers, learned with a neural network so that words used in similar contexts get similar vectors. The idea was introduced with Word2Vec (https://arxiv.org/abs/1301.3781; a classic introduction: https://jalammar.github.io/illustrated-word2vec/).

Word2Vec learns from sentences, split into words. NLTK's `sent_tokenize` splits a text into sentences. Run the cell below to see it on our example and the sentence that follows it in the speech.

In [ ]:
#Keep cell
two_sentences = example + " In words and deeds, we are showing the world that a new era of engagement has begun."
sentences = nltk.sent_tokenize(two_sentences)
print('Sentences:', sentences)
print('The words of the second sentence:', word_tokenize(sentences[1]))

The cell below does this for all speeches: over 63,000 sentences. Then it trains Word2Vec with the library Gensim, giving every word that appears at least five times a vector of 25 numbers. Run it, and you will see the vector for 'freedom'.

In [ ]:
#Keep cell
from gensim.models import Word2Vec

# every speech -> sentences -> lists of lowercase words
all_sentences = []
for speech in list(sotu_df['speechtext']):
    all_sentences.extend([word_tokenize(sentence.lower()) for sentence in nltk.sent_tokenize(speech)])
print(len(all_sentences), 'sentences')

# 25 numbers per word; words that appear fewer than 5 times are ignored
word2vec_model = Word2Vec(sentences=all_sentences, vector_size=25, min_count=5, workers=4, seed=4711)
word2vec_model.wv['freedom']

`most_similar` finds the words whose vectors are closest. Complete the cell below for the five words closest to 'freedom'. `topn` is the number of words.

In [ ]:
#Complete cell
similar_words = word2vec_model.wv.___('freedom', topn=5)
similar_words

Each result is a word and its similarity, from 0 (not similar) to 1. Now find the five words closest to the last word of our example, 'torture'. Do they make sense?

Count how often each word appears in the training sentences. Run the cell below.

In [ ]:
#Keep cell
print('freedom:', word2vec_model.wv.get_vecattr('freedom', 'count'))
print('torture:', word2vec_model.wv.get_vecattr('torture', 'count'))

> **🔍 What has the computer lost?** · Word2Vec knows a word only by its neighbours in these speeches. 'freedom' appears hundreds of times, and it is close to 'liberty' because presidents use them in similar sentences, not because the model knows what freedom is. 'torture' appears only nine times in 230 years of speeches: too few neighbours to learn from, so its closest words are close to random, and they change each time you train the model. Whatever associations the texts contain, the vectors contain too, and where the texts are silent, the vectors are noise. This is one reason why today's language models are trained on so much more text.

Word2Vec was the beginning of completely new approaches to text analysis, which have taken the world by storm with Transformers (https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)). In the workshop, we take the next step: language models that learn from far more text than these speeches.

---

## How the computer read our sentence

| Step | What the computer saw |
|---|---|
| The text | And that is why I can stand here tonight and say without exception or equivocation that the United States of America does not torture. |
| 1. Tokens | `['And', 'that', 'is', ..., 'United', 'States', 'of', 'America', 'does', 'not', 'torture', '.']` |
| 2. Cleaning | `['stand', 'tonight', 'say', ..., 'united', 'states', 'america', 'torture']`: "not" is gone |
| 3. Counting | one row of 23,495 numbers, almost all 0, in no particular order |
| 4. Trained model | "United States of America" is one place again, and the grammar keeps "not" |
| 5. Embeddings | each word is a vector, close to the words it shares contexts with |

### Check an agent

Ask Gemini to "find the ten most distinctive words of the 2008 State of the Union in sotu_df with tf-idf". Run its code in the cell below. Does it find the same words as we did? If not, which choices did it make differently: tokens, stopwords or weights? You can only check this because you did it yourself first.

In [ ]:
# Paste Gemini's code here, run it and compare

---

## Optional, at home

### Words that appear together

Which words go up and down together across the speeches? Pandas' `corrwith` (https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corrwith.html) correlates every column of `dtm_df` with one word: words that correlate tend to appear in the same speeches. This is the simplest version of the idea behind Word2Vec: you know a word by the company it keeps.

Run `dtm_df.corrwith(dtm_df['peace']).sort_values(ascending=False).head(10)` (1 is the highest correlation, 0 none), then do the same for 'war'.

Peace seems to be mainly a matter of war, and war goes with 'recommended' and 'expenditures'. Why might that be? Look at which speeches talk most about war.

### Bush's places on a map

`locations_spacy()`, defined in the next cell, geocodes the places spaCy found with OpenStreetMap (https://www.geoapify.com/openstreetmap-geocoding) and puts them on a map. It needs an internet connection and some patience. To save time, it maps only `doc[500:2500]`; use `locations_spacy(doc)` for the full speech. Ignore any warnings.

In [ ]:
#Keep cell

import geopandas as gpd
import geopy
from geopy.extra.rate_limiter import RateLimiter
import folium
from folium.plugins import FastMarkerCluster

def locations_spacy(doc):
    locator = geopy.geocoders.Nominatim(user_agent='mygeocoder')
    geocode = RateLimiter(locator.geocode, min_delay_seconds=1)
    locations = [ent for ent in doc.ents if ent.label_ in ['GPE']]
    locations_geocoded = []
    for l in locations:
        try:
            locations_geocoded.append(geocode(l).point)
        except:
            continue
    locations_geocoded = [(l[0], l[1]) for l in locations_geocoded]
    # Esri's street map needs no API key and works inside Colab
    fm = folium.Map(location=[38.305542, -30.384108], zoom_start = 2,
                    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
                    attr='Tiles &copy; Esri')
    FastMarkerCluster(data=locations_geocoded).add_to(fm)
    return fm

#locations_spacy(doc)
locations_spacy(doc[500:2500]) #doc[500:2500] for demonstration purposes. Use only doc for full document

Can you identify likely topics from the clusters of places in Bush's speech? Try it with another speech.